In [10]:
# Check the environment variables  and API keys
import os
from getpass import getpass

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    print(
        "python-dotenv is not installed. Please install it to load environment variables from a .env file."
    )

# Check for required environment variables
if not os.getenv("TAVILY_DEV_API_KEY"):
    os.environ["TAVILY_DEV_API_KEY"] = getpass("Enter your TAVILY_DEV_API_KEY: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter your PINECONE_API_KEY: ")

if not os.getenv("GROQ_DEV_API_KEY"):
    os.environ["GROQ_DEV_API_KEY"] = getpass("Enter your GROQ_DEV_API_KEY: ")


print("TAVILY_DEV_API_KEY configured: ", bool(os.getenv("TAVILY_DEV_API_KEY")))
print("PINECONE_API_KEY configured: ", bool(os.getenv("PINECONE_API_KEY")))
print("GROQ_DEV_API_KEY configured: ", bool(os.getenv("GROQ_DEV_API_KEY")))

TAVILY_DEV_API_KEY configured:  True
PINECONE_API_KEY configured:  True
GROQ_DEV_API_KEY configured:  True


### Load Sample Document with LangGraph


In [11]:
from langchain_community.document_loaders import WebBaseLoader

WEB_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_path=WEB_URL, requests_kwargs={"headers": {"User-Agent": "Mozilla/5.0"}}
)

raw_documents = loader.load()

print(f"Loaded {len(raw_documents)} documents from {WEB_URL}")
print(f"Source URL: {raw_documents[0].metadata.get('source')}")
print(f"Document content preview: {raw_documents[0].page_content[:500]}...")

Loaded 1 documents from https://docs.langchain.com/oss/python/langgraph/agentic-rag
Source URL: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Document content preview: 




Build a custom RAG agent with LangGraph - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutor...


In [12]:
# Split the documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150, add_start_index=True
)

chunks = splitter.split_documents(raw_documents)

print(f"Split into {len(chunks)} chunks.")
print(f"First chunk preview: {chunks[0].page_content[:900]}...")

Split into 29 chunks.
First chunk preview: Build a custom RAG agent with LangGraph - Docs by LangChain...


In [13]:
# Create embeddings for the chunks
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)


sample_vector = embeddings.embed_query("What was will the next revolution in AI be?")
print(f"Sample embedding vector length: {len(sample_vector)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3295.70it/s]


Sample embedding vector length: 384


###### sample_vector


#### Create a vector store using Pinecone


In [14]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = "homa-rag-knowledge-base"
NAMESPACE_NAME = "langchain-agentic-rag"

# Initialize Pinecone client
pinecone_client = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))


# Check if the index exists, if not create it
index_names = pinecone_client.list_indexes().names()
if INDEX_NAME not in index_names:
    print(f"Creating index '{INDEX_NAME}'...")
    pinecone_client.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# Wait for the index to be ready, whether it was just created or already existed
while True:
    index_description = pinecone_client.describe_index(INDEX_NAME)
    if index_description.status["ready"]:
        break
    print("Waiting for the index to be ready...")
    time.sleep(5)

print(f"Index '{INDEX_NAME}' is ready.")


Index 'homa-rag-knowledge-base' is ready.


In [15]:
# Upload the document chunks and create the vector store
document_ids = [
    f"{document.metadata.get('source', 'unknown')}:{document.metadata.get('start_index', i)}"
    for i, document in enumerate(chunks)
]

vectorestore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=document_ids,
    index_name=INDEX_NAME,
    namespace=NAMESPACE_NAME,
)

retriever = vectorestore.as_retriever(
    search_kwargs={"k": 4, "namespace": NAMESPACE_NAME}
)

print(f"Retriever created with {retriever.search_kwargs['k']} results per query.")

Retriever created with 4 results per query.


##### Test the retriever with a sample query


In [16]:
query = "What are some important features of the Agentic RAG framework?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i + 1}:")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Content preview: {doc.page_content[:700]}...")


Document 1:
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Content preview: Build a custom RAG agent with LangGraph - Docs by LangChain...

Document 2:
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Content preview: Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual ...

Document 3:
Source: https://docs.lan

In [17]:
# # TODO: Use with caution
# pinecone_client.Index(INDEX_NAME).delete(
#     delete_all=True,
#     namespace=NAMESPACE_NAME,
# )